# Extractor Pipeline — Verification Report (Read‑Only)

This notebook reads existing artifacts (Stages 04–09) and prints concise metrics and PASS/FAIL checks.
No pipeline logic is executed; it only reads JSON files and the environment.

In [ ]:
from pathlib import Path
import json
import os

RUN_ROOT = Path('data/results/pipeline')
ART = {
  'sections': RUN_ROOT / '04_section_builder/json_output/04_sections.json',
  'tables': RUN_ROOT / '05_table_extractor/json_output/05_tables.json',
  'figures': RUN_ROOT / '06_figure_extractor/json_output/06_figures.json',
  'enriched_tables': RUN_ROOT / '06a_title_caption_enricher/json_output/05_tables.enriched.json',
  'enriched_figures': RUN_ROOT / '06a_title_caption_enricher/json_output/06_figures.enriched.json',
  'reflowed': RUN_ROOT / '07_reflow_section/json_output/07_reflowed.json',
  'reqs': RUN_ROOT / '07_requirements_miner/json_output/07_requirements.json',
  'theorems': RUN_ROOT / '08_lean4_theorem_prover/json_output/08_theorems.json',
}

def _read_json(p: Path):
    try:
        return json.loads(p.read_text()) if p.exists() else None
    except Exception as e:
        print('read-failed', p, e)
        return None


## Quick Slices

In [ ]:
sec = _read_json(ART['sections'])
rf = _read_json(ART['reflowed'])
rq = _read_json(ART['reqs'])
th = _read_json(ART['theorems'])
print('sections:', bool(sec))
print('reflow:', rf.get('status') if isinstance(rf, dict) else rf)
print('requirements:', len((rq or {}).get('requirements', [])) if isinstance(rq, dict) else 0)
print('proofs.stats:', (th or {}).get('statistics'))


## Verification Checklist

In [ ]:
from pathlib import Path as _P

summary = {"stages": {}, "env": {}, "warnings": []}

# env
base = os.environ.get('CHUTES_API_BASE', '')
key = os.environ.get('CHUTES_API_KEY', '')
summary['env'] = {
    'CHUTES_API_BASE_ok': base.endswith('/v1'),
    'CHUTES_API_KEY_present': len(key) > 10,
}
print('CHUTES_API_BASE_ok=', summary['env']['CHUTES_API_BASE_ok'])
print('CHUTES_API_KEY_present=', summary['env']['CHUTES_API_KEY_present'])

et = _read_json(ART['enriched_tables'])
ef = _read_json(ART['enriched_figures'])
summary['stages']['06a_enriched'] = bool(isinstance(et, list) and et and isinstance(ef, list) and ef)
print('06a_enriched=', summary['stages']['06a_enriched'])

# 07
rf = _read_json(ART['reflowed']) or {}
summary['stages']['07_completed'] = rf.get('status') == 'Completed' and (rf.get('section_count') or 0) > 0
src_files = rf.get('source_files') or {}
summary['stages']['07_consumed_enriched'] = str(src_files.get('tables', '')).endswith('05_tables.enriched.json')
print('07_completed=', summary['stages']['07_completed'])

rq = _read_json(ART['reqs']) or {}
reqs = rq.get('requirements', rq if isinstance(rq, list) else [])
weak = [r for r in reqs if isinstance(r, dict) and str(r.get('strength', '')).lower() == 'weak']
summary['stages']['07_requirements'] = {'count': len(reqs), 'strict_modal': len(weak) == 0}
print('07_requirements.count=', len(reqs))
print('07_requirements.strict=', len(weak) == 0)

# 08
th = _read_json(ART['theorems']) or {}
st = th.get('statistics') or {}
summary['stages']['08_proofs'] = {
    'present': bool(st),
    'successful_proofs': st.get('successful_proofs'),
    'total': st.get('total_requirements_found'),
}
print('08_proofs=', summary['stages']['08_proofs'])

# save
_P('scripts/artifacts').mkdir(parents=True, exist_ok=True)
(_P('scripts/artifacts') / 'pipeline_verification.json').write_text(json.dumps(summary, indent=2))
summary
